# RAG benchmark · native DeepEval + thin cli-agent target adapter

The target integration lives entirely in `scripts/benchmark_adapter.py`.
The target sees **only each golden question**. The separate DeepEval judge
receives the reference answer, plus verified selected Knowledge passages
when measuring retrieval. Judge credentials must be approved for this data.

## 1. Target and independent judge configuration

Use the cli-agent's ordinary model/config for the target. The judge is explicitly
configured independently, including its own provider and credentials.
Store API keys in your environment, never in notebook cells.

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd().resolve()
GOLDENS_FILE = WORKSPACE / "rag-goldens" / "goldens.json"
OUTPUT_DIR = WORKSPACE / "rag-goldens" / "evaluation"

# Target cli-agent SDK settings
CONFIG_FILE = None
ADMIN_CONFIG_FILE = None
APPROVE_TOOLS = ()
CAPTURE_KNOWLEDGE_CONTENT = True
MAX_CASES = 3                 # None = all goldens
SDK_TIMEOUT_SECONDS = 600.0
FRESH_SDK = False

# Completely independent DeepEval judge
JUDGE_PROVIDER = "openrouter" # openrouter / openai / ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False

## 2. Execute target questions (reusable answer checkpoints)

Each golden uses a fresh cli-agent instance. Cached successful SDK responses
are reused unless the SDK config fingerprint changes or FRESH_SDK is true.
Live index/model/external tool changes require FRESH_SDK=true manually.

In [ ]:
sys.path.insert(0, str(WORKSPACE / "flow/rag/generate_goldens/scripts"))
from benchmark_adapter import (
    load_goldens, sdk_target_factory, target_fingerprint, collect_responses,
    deep_eval_cases, judge_model, benchmark_metrics, source_recall,
)

goldens = load_goldens(GOLDENS_FILE, MAX_CASES)
factory = sdk_target_factory(
    WORKSPACE, CONFIG_FILE, ADMIN_CONFIG_FILE,
    tuple(APPROVE_TOOLS), CAPTURE_KNOWLEDGE_CONTENT,
)
target_sha = target_fingerprint(
    WORKSPACE, CONFIG_FILE, ADMIN_CONFIG_FILE,
    tuple(APPROVE_TOOLS), CAPTURE_KNOWLEDGE_CONTENT,
)
responses = await collect_responses(
    goldens, target_factory=factory, output=OUTPUT_DIR,
    target_fingerprint=target_sha, timeout=SDK_TIMEOUT_SECONDS,
    fresh=FRESH_SDK,
)
print(f"{len(responses)} target answers ready")

## 3. Native DeepEval dataset and judge

The adapter creates actual `LLMTestCase` objects in an `EvaluationDataset`.
`retrieval_context` is populated only from verified selected Knowledge text,
never from source quotes included in the golden.

In [ ]:
dataset = deep_eval_cases(goldens, responses)
judge = judge_model(JUDGE_PROVIDER, JUDGE_MODEL, JUDGE_BASE_URL)
print(f"{len(dataset.test_cases)} cases; "
      f"{sum(bool(c.retrieval_context) for c in dataset.test_cases)} with retrieval")
print(f"Judge: {JUDGE_PROVIDER} / {JUDGE_MODEL}")

## 4. DeepEval handles scoring, concurrency, caching and reports

Answer metrics run on every case. Retrieval metrics run separately on the
subset with actual selected Knowledge passages, avoiding fabricated
grounding and missing-context errors.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

options = {
    "cache_config": CacheConfig(
        use_cache=not FRESH_EVALUATION, write_cache=True,
    ),
    "async_config": AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
    ),
    "hyperparameters": {
        "target_sha256": target_sha,
        "judge": f"{JUDGE_PROVIDER}/{JUDGE_MODEL}",
    },
}
answer_results = evaluate(
    test_cases=dataset,
    metrics=benchmark_metrics(judge, THRESHOLD, retrieval=False),
    identifier="rag-answer-quality",
    **options,
)

retrieval_cases = [case for case in dataset.test_cases
                   if case.retrieval_context]
retrieval_results = None
if retrieval_cases:
    retrieval_results = evaluate(
        test_cases=retrieval_cases,
        metrics=benchmark_metrics(judge, THRESHOLD, retrieval=True)[2:],
        identifier="rag-retrieval-quality",
        **options,
    )
else:
    print("No observed selected Knowledge context; retrieval metrics skipped.")

## 5. Optional cli-agent-specific source recall

This is not a DeepEval judge score. It reports whether expected source
documents were found in initial search, in all candidates or selected
for the final prompt. Paths must match exactly.

In [ ]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame([
    {"golden_id": g["id"], **(source_recall(g, responses[g["id"]]) or {})}
    for g in goldens
]))
print("Native DeepEval reports:", OUTPUT_DIR / "deepeval")

## Operational notes

DeepEval owns metric caching and reports. The small adapter caches only
target responses, so changing JUDGE_MODEL never reruns the target.
FRESH_EVALUATION disables DeepEval cache reuse; FRESH_SDK forces target runs.
DeepEval G-Eval requires a compatible judge model (not all reasoning models
support the necessary scoring/log-probability behavior).
Keep saved Knowledge traces and judge data private.